## need a BW2 not BW25 venv, so that can later export as bw2datapackage

### here change 1. `cf_point = xr.open_dataset('data/v2026/fixedCO2_CF_GWP1_100_perSSP_MY_majorghgs_ModC.nc')` for either fixed or dpCO2 approach, then rerun all cells to create new method; 2.  gwp_method = "with dp CO2" or "with fixed CO2"

In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append('/Users/susierwu/dpLCA_main/') 
from utils import *
from utils.newbw2method_dpLCIA import * 

In [4]:
bw2data.projects.set_current('iveo_v1')

In [5]:
len(list([m for m in bw2data.methods if  'SSP' in str(m)]))

18

In [6]:
mybio = bw2data.Database("biosphere3")
len(mybio)

4709

### read in pre-calculated .nc LCIA dataset

In [8]:
cf_point = xr.open_dataset('data/v2026/dpCO2_CF_GWP_1_100_perSSP_MY_majorghgs_ModC.nc')
cf_point

<xarray.Dataset> Size: 44kB
Dimensions:    (Year: 100, SSP: 6, ModelYear: 3)
Coordinates:
  * Year       (Year) int32 400B 1 2 3 4 5 6 7 8 9 ... 93 94 95 96 97 98 99 100
  * SSP        (SSP) object 48B '119' '126' '245' '434' '460' '585'
  * ModelYear  (ModelYear) int32 12B 2030 2040 2050
Data variables:
    CO2_GWP    (SSP, ModelYear, Year) float64 14kB ...
    CH4_GWP    (SSP, ModelYear, Year) float64 14kB ...
    N2O_GWP    (SSP, ModelYear, Year) float64 14kB ...

### read in premise_GWP, all minor_ghg using static amount 

In [9]:
prem_gwp_dfraw = pd.read_excel("premise_gwp/lcia_gwp2021_100a_w_bio.xlsx")
prem_gwp_dfraw.head()

,name,categories,amount
0,Bromopropane,air::unspecified,0.052
1,Butane,air::urban air close to ground,0.006
2,Butane,air::non-urban air or from high stacks,0.006
3,Butane,"air::low population density, long-term",0.006
4,Butane,air::lower stratosphere + upper troposphere,0.006


### calling the assign_dpGWP class, see what it looks like for final CF

In [10]:
xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = '119', fairMY = 2030 )
minorg, allg = xx.get_minorand_allGHG()
cc = xx.prep_empty_C(allg)
#cc.head()
pcc = xx.assign_minorghg_to_C_GWP100(cc, minorg)
fcc = xx.assign_majorghg_dCC(pcc)
fcc

,Bromopropane,Butane,"Carbon monoxide, fossil","Carbon monoxide, from soil or biomass stock","Carbon monoxide, non-fossil",Chloroform,Ethane,"Ethane, 1,1,1,2-tetrafluoro-, HFC-134a","Ethane, 1,1,1-trichloro-, HCFC-140","Ethane, 1,1,1-trifluoro-, HFC-143a",...,"Carbon dioxide, non-fossil","Carbon dioxide, in air","Carbon dioxide, to soil or biomass stock","Carbon dioxide, from soil or biomass stock","Carbon dioxide, fossil","Carbon dioxide, non-fossil, resource correction","Methane, from soil or biomass stock","Methane, fossil","Methane, non-fossil",Dinitrogen monoxide
GWP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,137.246894,137.246894,137.246894,180.979604
GWP2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,136.641821,136.641821,136.641821,186.331567
GWP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,135.332017,135.332017,135.332017,190.469524
GWP4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,133.445501,133.445501,133.445501,194.059480
GWP5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,131.122994,131.122994,131.122994,196.879089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GWP96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,29.703744,29.703744,29.703744,183.086555
GWP97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,29.444140,29.444140,29.444140,182.503756
GWP98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,29.189287,29.189287,29.189287,181.935829
GWP99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,-1.0,-1.0,1.0,1.0,-1.0,28.939059,28.939059,28.939059,181.369764


### run MY2030/2040&2050 and all SSP together

In [11]:
for mmy in [2030, 2040, 2050]: 
    for sp in ['119', '126', '245', '434', '460', '585']:  
        xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = sp, fairMY = mmy, GWP100_only = True )
        minorg, allg = xx.get_minorand_allGHG()
        emt_C =  xx.prep_empty_C(allg)
        fullminor_C = xx.assign_minorghg_to_C_GWP100(emt_C, minorg )
        #print(fullminor_C.head() )
        full_allC = xx.assign_majorghg_dCC(fullminor_C)
        data = xx.prep_data_for_bw2method (full_allC, mybio = bw2data.Database("biosphere3"))
        print(len(data))
        xx.prep_final_dCC_bw2method (data, gwp_method = "with dp CO2")

start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2030
1
 start creating new method SSP119 MY2030, with with dp CO2 approach 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 with dp CO2')
start preparing data to be assigned as new bw2data.methods, for SSP 126 and fair_MY2030
1
 start creating new method SSP126 MY2030, with with dp CO2 approach 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP126', 'MY2030', 'pGWP100 with dp CO2')
start preparing data to be assigned as new bw2data.methods, for SSP 245 and fair_MY2030
1
 start creating new method SSP245 MY2030, with with dp CO2 approach 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP245', 'MY2030', 'pGWP100 with dp CO2')
start preparing data to be assigned as new bw2data.methods, for SSP 434 and fair_MY2030
1
 start creating new method SSP434 MY2030, with wi

In [12]:
len(list([m for m in bw2data.methods if 'pGWP100' in str(m)]))

36